In [2]:
from datasets import load_dataset, Dataset, DatasetDict
import random
import datasets

### Sample Traning Instances for Differential Input Generation

Call the function by specifying the dataset_path to the huggingface dataset that you would like to sample the training instances from.

In [16]:
def sample_adversarial_training_instances(dataset_path):
    random.seed(42)
    dataset = load_dataset(dataset_path, split='train')
    sample_size = int(0.1*len(dataset))
    sampled_indices = random.sample(range(len(dataset)), sample_size)
    sampled_dataset = dataset.select(sampled_indices)
    push_path = dataset_path.split("/")[1] + "_adversarial_training"
    sampled_dataset.push_to_hub(push_path, private=False) # , private=True
    print("Finished")

After sampling, you can find the dataset in your huggingface repo that ends with "adversarial_training", follow the instructions in the repository README file to generate differential inputs.

### Mix Adversarial Training Data

With differential inputs generated, the next step is to mix the differential inputs with the original training dataset to create the adversarial training dataset.

In [11]:
from datasets import load_dataset, DatasetDict, concatenate_datasets
import random
import datasets

def combine_datasets_adv(path_to_adv, path_to_ori, path_to_save):
    dataset_adv = load_dataset(path_to_adv, split='train')
    dataset_ori = load_dataset(path_to_ori)
    dataset_adv = dataset_adv.rename_column("text", "sentence")
    dataset_adv = dataset_adv.cast(dataset_ori['train'].features)

    # Combine the training sets
    combined_train = concatenate_datasets([dataset_adv, dataset_ori['train']])
   
    # Shuffle the combined dataset
    combined_train = combined_train.shuffle(seed=42)

    # Create a new DatasetDict
    combined_dataset = DatasetDict({
        'train': combined_train,
        'validation': dataset_ori['validation'],
        'test': dataset_ori['test']
    })

    # Push the new dataset to Hugging Face
    combined_dataset.push_to_hub(path_to_save)
    print("Finished")

The combined dataset can now be used for adversarial training.